In [ ]:
#@title  Explicit Adjoint Exposure (EAE)

import os, math, time, json, random
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import IterableDataset, DataLoader

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > T4 GPU"
device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu_name}  (compute capability {cc_major}.{cc_minor})")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

BF16_OK = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print(f"Mixed precision dtype selected: {AMP_DTYPE}  (GradScaler {'disabled' if BF16_OK else 'enabled'})")

SEED = 1337
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------------------------
# Colab form params — the knobs you actually want to change run-to-run.
# ---------------------------------------------------------------------------
MODEL_SIZE = "small"  #@param ["tiny", "small", "base", "custom"]
MAX_STEPS = 800       #@param {type:"integer"}
SAMPLE_EVERY_STEPS = 250  #@param {type:"integer"}
EVAL_EVERY_STEPS = 200   #@param {type:"integer"}
SAVE_EVERY_STEPS = 500  #@param {type:"integer"}
EVAL_BATCHES = 20  #@param {type:"integer"}
GENERATION_PROMPTS = "The history of artificial intelligence,Once upon a time,In order to solve this problem,"  #@param {type:"string"}
MAX_NEW_TOKENS = 60  #@param {type:"integer"}
RESUME_FROM_CHECKPOINT = ""  #@param {type:"string"}

GENERATION_PROMPTS = [p.strip() for p in GENERATION_PROMPTS.split(",") if p.strip()]

@dataclass
class ModelConfig:
    vocab_size: int = 49152          # overwritten once the tokenizer loads
    hidden_size: int = 512
    n_layers: int = 12
    n_heads: int = 8
    n_kv_heads: int = 4              # GQA: n_heads must be divisible by n_kv_heads
    intermediate_size: int = 1408    # SwiGLU inner dim (~2.67x hidden, rounded to 64)
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-5
    tie_word_embeddings: bool = True
    dropout: float = 0.0
    use_qk_norm: bool = True
    use_gradient_checkpointing: bool = False  # <--- ALLWAYS FALSE (We do this naturally now)

PRESETS = {
    # ~18M total params (~5.6M non-embedding) -- fastest option, good for smoke-testing the pipeline
    "tiny":  dict(hidden_size=256,  n_layers=8,  n_heads=8,  n_kv_heads=2, intermediate_size=704,  max_seq_len=1024),
    # ~60M total params (~35M non-embedding) -- SmolLM2-135M-ish recipe at a T4-friendly size
    "small": dict(hidden_size=512,  n_layers=12, n_heads=8,  n_kv_heads=4, intermediate_size=1408, max_seq_len=1024),
    # ~140M total params (~100M non-embedding) -- fits a T4 with grad checkpointing + small micro-batch
    "base":  dict(hidden_size=768,  n_layers=16, n_heads=12, n_kv_heads=4, intermediate_size=2048, max_seq_len=1024),
}

if MODEL_SIZE == "custom":
    # ------ set your own architecture here ------
    custom_cfg = dict(hidden_size=640, n_layers=14, n_heads=10, n_kv_heads=2,
                       intermediate_size=1728, max_seq_len=1024)
    model_cfg = ModelConfig(**custom_cfg)
else:
    model_cfg = ModelConfig(**PRESETS[MODEL_SIZE])

assert model_cfg.hidden_size % model_cfg.n_heads == 0, "hidden_size must be divisible by n_heads"
assert model_cfg.n_heads % model_cfg.n_kv_heads == 0, "n_heads must be divisible by n_kv_heads (GQA)"
print(model_cfg)

@dataclass
class TrainConfig:
    # dataset
    dataset_name: str = "openbmb/Ultra-FineWeb"
    dataset_split: str = "en"          # English subset (1.16B rows); HF split name, not a config name
    shuffle_buffer_size: int = 10_000

    # optimization
    micro_batch_size: int = 8          # per-forward batch size (lower this first if you OOM)
    grad_accum_steps: int = 8          # effective batch = micro_batch_size * grad_accum_steps
    max_steps: int = MAX_STEPS         # optimizer steps
    warmup_steps: int = 200
    max_lr: float = 4e-4
    min_lr_ratio: float = 0.1          # cosine floor = max_lr * min_lr_ratio
    weight_decay: float = 0.1
    betas: tuple = (0.9, 0.95)
    grad_clip: float = 1.0

    # evaluation
    eval_batches: int = EVAL_BATCHES   # number of held-out (x, y) batches used for eval loss
    eval_every: int = EVAL_EVERY_STEPS
    sample_every: int = SAMPLE_EVERY_STEPS
    sample_prompts: tuple = tuple(GENERATION_PROMPTS)
    max_new_tokens: int = MAX_NEW_TOKENS

    # infra
    compile_model: bool = False             # <--- SET TO FALSE (Torch compile cannot trace our shattered graph)
    log_every: int = 20
    save_every: int = SAVE_EVERY_STEPS
    checkpoint_dir: str = "/content/checkpoints"

train_cfg = TrainConfig()
os.makedirs(train_cfg.checkpoint_dir, exist_ok=True)

effective_batch_tokens = train_cfg.micro_batch_size * train_cfg.grad_accum_steps * model_cfg.max_seq_len
print(f"Effective batch size: {train_cfg.micro_batch_size * train_cfg.grad_accum_steps} sequences "
      f"({effective_batch_tokens:,} tokens/step)")
print(f"Planned training budget: {train_cfg.max_steps * effective_batch_tokens:,} tokens total")
print(f"Sampling every {train_cfg.sample_every} steps | Eval every {train_cfg.eval_every} steps "
      f"({train_cfg.eval_batches} held-out batches) | Checkpointing every {train_cfg.save_every} steps")

# ============================================================================
# Model
# ============================================================================

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x.to(dtype) * self.weight


def precompute_rope(head_dim, max_seq_len, theta, device):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, cos, sin):
    cos = cos[None, None, :, :].to(q.dtype)
    sin = sin[None, None, :, :].to(q.dtype)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class GQAAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.hidden_size // cfg.n_heads
        self.n_rep = cfg.n_heads // cfg.n_kv_heads
        self.dropout = cfg.dropout

        self.q_proj = nn.Linear(cfg.hidden_size, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.hidden_size, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.hidden_size, cfg.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * self.head_dim, cfg.hidden_size, bias=False)

        self.use_qk_norm = cfg.use_qk_norm
        if self.use_qk_norm:
            self.q_norm = RMSNorm(self.head_dim, cfg.rms_norm_eps)
            self.k_norm = RMSNorm(self.head_dim, cfg.rms_norm_eps)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if self.use_qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        q, k = q.transpose(1, 2), k.transpose(1, 2)
        q, k = apply_rope(q, k, cos[:T], sin[:T])

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.up_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.down_proj = nn.Linear(cfg.intermediate_size, cfg.hidden_size, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.attn = GQAAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.mlp(self.mlp_norm(x))
        return x


class SOTALM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        if cfg.tie_word_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        head_dim = cfg.hidden_size // cfg.n_heads
        cos, sin = precompute_rope(head_dim, cfg.max_seq_len, cfg.rope_theta, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        # NOTE: self.apply() below will visit both tok_emb and lm_head, but since
        # tie_word_embeddings makes them the *same tensor*, it just gets normal_()
        # initialized twice in a row with identical stats -- harmless, not a bug,
        # but worth calling out so it doesn't look like a double-init mistake.
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = self.rope_cos.to(x.device)
        sin = self.rope_sin.to(x.device)
        for blk in self.blocks:
            if self.cfg.use_gradient_checkpointing and self.training:
                x = checkpoint(blk, x, cos, sin, use_reentrant=False)
            else:
                x = blk(x, cos, sin)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)).float(), targets.view(-1), ignore_index=-100
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=64, temperature=0.8, top_k=50):
        was_training = self.training
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        if was_training:
            self.train()
        return idx


# ============================================================================
# Tokenizer
# ============================================================================

from transformers import AutoTokenizer

TOKENIZER_CANDIDATES = ["HuggingFaceTB/SmolLM2-135M", "gpt2"]

tokenizer = None
for name in TOKENIZER_CANDIDATES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(name)
        print(f"Loaded tokenizer: {name}  (vocab_size={tokenizer.vocab_size})")
        break
    except Exception as e:
        print(f"Could not load {name}: {e}")

assert tokenizer is not None, "No tokenizer could be loaded."
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.eos_token_id is None:
    tokenizer.add_special_tokens({"eos_token": "<|endoftext|>"})

model_cfg.vocab_size = len(tokenizer)
print("model_cfg.vocab_size updated ->", model_cfg.vocab_size)

# ============================================================================
# Model instantiation + sanity check
# ============================================================================

model = SOTALM(model_cfg).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_embed_params = model.tok_emb.weight.numel()
print(f"Total parameters:          {n_params:,}")
print(f"  of which embedding/head: {n_embed_params:,} (tied)")
print(f"  non-embedding params:    {n_params - n_embed_params:,}")
print(f"Approx AdamW state memory: {n_params * 8 / 1e9:.2f} GB")

model.train()
_dummy = torch.randint(0, model_cfg.vocab_size, (2, 32), device=device)
_logits, _loss = model(_dummy, _dummy)
_loss.backward()
model.zero_grad(set_to_none=True)
print(f"Sanity check OK. Dummy loss (~ln(vocab_size)={math.log(model_cfg.vocab_size):.2f}): {_loss.item():.2f}")

# ============================================================================
# Data: streaming train set + a small fixed held-out eval set
# ============================================================================

from datasets import load_dataset

def make_hf_stream(seed=SEED):
    ds = load_dataset(train_cfg.dataset_name, split=train_cfg.dataset_split, streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=train_cfg.shuffle_buffer_size)
    return ds


class PackedTokenDataset(IterableDataset):
    '''Streams raw text, tokenizes on the fly, and packs tokens into fixed-length
    (seq_len + 1) chunks with no padding. Each yielded sample is used as
    input_ids = chunk[:-1], targets = chunk[1:] (standard next-token-prediction shift).'''

    def __init__(self, tokenizer, seq_len, seed=SEED):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.seed = seed

    def __iter__(self):
        buffer = []
        while True:  # loop forever; Ultra-FineWeb-en has ~1.16B rows, far more than we need
            stream = make_hf_stream(seed=self.seed)
            for example in stream:
                text = example.get("content", "")
                if not text:
                    continue
                ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
                ids.append(self.tokenizer.eos_token_id)
                buffer.extend(ids)
                while len(buffer) >= self.seq_len + 1:
                    chunk = buffer[: self.seq_len + 1]
                    buffer = buffer[self.seq_len:]
                    x = torch.tensor(chunk[:-1], dtype=torch.long)
                    y = torch.tensor(chunk[1:], dtype=torch.long)
                    yield x, y


def collate(batch):
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.stack(ys)


train_dataset = PackedTokenDataset(tokenizer, model_cfg.max_seq_len, seed=SEED)
train_loader = DataLoader(train_dataset, batch_size=train_cfg.micro_batch_size,
                           collate_fn=collate, num_workers=0)

_it = iter(train_loader)
_xb, _yb = next(_it)
print("batch shapes:", _xb.shape, _yb.shape)
print("sample decoded text:", tokenizer.decode(_xb[0][:60]))
del _it, _xb, _yb


def build_eval_set(n_batches):
    '''Pulls a small, fixed set of batches from a differently-seeded shuffle of
    the same split and caches them on CPU. This is a lightweight held-out set
    for tracking generalization during a short Colab run -- with a streaming
    dataset there's no clean deterministic train/val file split available, so
    a distinct shuffle seed is the practical way to reduce (not guarantee-zero)
    overlap with the training stream.'''
    eval_ds = PackedTokenDataset(tokenizer, model_cfg.max_seq_len, seed=SEED + 1)
    eval_loader = DataLoader(eval_ds, batch_size=train_cfg.micro_batch_size, collate_fn=collate, num_workers=0)
    it = iter(eval_loader)
    batches = []
    for _ in range(n_batches):
        xb, yb = next(it)
        batches.append((xb, yb))
    del it
    return batches


print(f"Building held-out eval set ({train_cfg.eval_batches} batches)...")
eval_batches_cache = build_eval_set(train_cfg.eval_batches)
print("Eval set ready.")


@torch.no_grad()
def evaluate():
    model.eval()
    total_loss, n = 0.0, 0
    for xb, yb in eval_batches_cache:
        xb, yb = xb.to(device), yb.to(device)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            _, loss = model(xb, yb)
        total_loss += loss.item()
        n += 1
    model.train()
    avg_loss = total_loss / max(n, 1)
    ppl = math.exp(min(avg_loss, 20))
    return avg_loss, ppl

# ============================================================================
# Optimizer, schedule, checkpoints
# ============================================================================

def configure_optimizer(model, weight_decay, lr, betas):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() >= 2:
            decay.append(p)
        else:
            no_decay.append(p)
    groups = [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    use_fused = torch.cuda.is_available()
    return torch.optim.AdamW(groups, lr=lr, betas=betas, eps=1e-8, fused=use_fused)


def get_lr(step, cfg: TrainConfig):
    if step < cfg.warmup_steps:
        return cfg.max_lr * (step + 1) / cfg.warmup_steps
    if step >= cfg.max_steps:
        return cfg.max_lr * cfg.min_lr_ratio
    progress = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return cfg.max_lr * cfg.min_lr_ratio + coeff * cfg.max_lr * (1 - cfg.min_lr_ratio)


optimizer = configure_optimizer(model, train_cfg.weight_decay, train_cfg.max_lr, train_cfg.betas)
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

start_step = 0
if RESUME_FROM_CHECKPOINT and os.path.exists(RESUME_FROM_CHECKPOINT):
    print(f"Resuming from {RESUME_FROM_CHECKPOINT}")
    ckpt = torch.load(RESUME_FROM_CHECKPOINT, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_step = ckpt["step"]
    print(f"Resumed at step {start_step}")

compiled_model = model
if train_cfg.compile_model:
    try:
        compiled_model = torch.compile(model)
        print("torch.compile enabled.")
    except Exception as e:
        print(f"torch.compile unavailable ({e}); continuing without it.")
        compiled_model = model


def save_checkpoint(step):
    path = os.path.join(train_cfg.checkpoint_dir, f"step_{step}.pt")
    torch.save({
        "step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_cfg": asdict(model_cfg),
        "train_cfg": {k: v for k, v in asdict(train_cfg).items()},
    }, path)
    print(f"[checkpoint] saved {path}")


@torch.no_grad()
def sample(prompts=None, max_new_tokens=None):
    prompts = prompts or list(train_cfg.sample_prompts)
    max_new_tokens = max_new_tokens or train_cfg.max_new_tokens
    print("--- samples ---")
    for prompt in prompts:
        ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=0.8, top_k=50)
        print(f"[{prompt!r}] -> {tokenizer.decode(out[0], skip_special_tokens=True)}")
    print("---------------")


# ============================================================================
# TRAINING LOOP
# ============================================================================

data_iter = iter(train_loader)
model.train()
t0 = time.time()
running_loss = 0.0
history = {"step": [], "train_loss": [], "eval_step": [], "eval_loss": [], "eval_ppl": [], "peak_vram_gb": []}
torch.cuda.reset_peak_memory_stats(device)

for step in range(start_step, train_cfg.max_steps):
    lr = get_lr(step, train_cfg)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0
    
    for micro_step in range(train_cfg.grad_accum_steps):
        xb, yb = next(data_iter)
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

        # # -------------------------------------------------------------------
        # # 1. PRIMAL SWEEP (Shattering the Global Chain Rule)
        # # -------------------------------------------------------------------
        # primal_states = []
        # with torch.no_grad():
        #     x = model.tok_emb(xb)
        #     primal_states.append(x)
            
        #     cos = model.rope_cos.to(device)
        #     sin = model.rope_sin.to(device)
            
        #     for blk in model.blocks:
        #         x = blk(x, cos, sin)
        #         primal_states.append(x)
            
        #     # Note: We don't run final_norm and lm_head here because we will
        #     # combine them with the Dual Boundary condition computation below 
        #     # to be mathematically precise.
            
        #     # Forward pass just to calculate standard Loss for your logging
        #     x_norm_log = model.final_norm(x)
        #     logits_log = model.lm_head(x_norm_log)
        #     loss_val = F.cross_entropy(logits_log.view(-1, logits_log.size(-1)).float(), yb.view(-1), ignore_index=-100)
        #     step_loss += loss_val.item() / train_cfg.grad_accum_steps

        # # -------------------------------------------------------------------
        # # 2. DUAL BOUNDARY CONDITION
        # # -------------------------------------------------------------------
        # probs = F.softmax(logits_log.float(), dim=-1)
        # valid_mask = (yb != -100).unsqueeze(-1).float()
        # yb_safe = torch.where(yb == -100, torch.zeros_like(yb), yb)
        # targets_one_hot = F.one_hot(yb_safe, num_classes=logits_log.size(-1)).float()
        
        # # Cross Entropy Dual Momentum (Derivation: p - y)
        # N = max(1, (yb != -100).sum().item())
        # # We divide by grad_accum_steps here so gradients sum up perfectly across micro-batches
        # d_logits = ((probs - targets_one_hot) * valid_mask) / (N * train_cfg.grad_accum_steps)
        
        # # Mixed Precision Handling: If using FP16, we MUST scale the momentum here
        # # so that our manual local gaps don't underflow.
        # if AMP_DTYPE == torch.float16:
        #     d_logits = d_logits * scaler.get_scale()

        # # Step back through Final Norm and LM Head locally
        # x_last = primal_states[-1].detach().requires_grad_(True)
        # with torch.enable_grad():
        #     with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
        #         x_norm_val = model.final_norm(x_last)
        #         logits_val = model.lm_head(x_norm_val)
            
        #     # Fenchel Gap Proxy for the Head
        #     head_gap = torch.sum(logits_val.float() * d_logits.detach())
        
        # # Calculate local dual momentum (d_current) and parameter gradients
        # head_params = list(model.final_norm.parameters()) + list(model.lm_head.parameters())
        # head_grads = torch.autograd.grad(head_gap, [x_last] + head_params)
        
        # d_current = head_grads[0] # The momentum entering the top transformer block
        
        # # Accumulate head gradients
        # for p, g in zip(head_params, head_grads[1:]):
        #     if p.grad is None: p.grad = g.clone()
        #     else: p.grad += g

        # -------------------------------------------------------------------
        # 1. PRIMAL SWEEP (Shattering the Global Chain Rule)
        # -------------------------------------------------------------------
        primal_states = []
        with torch.no_grad():
            x = model.tok_emb(xb)
            primal_states.append(x)
            
            cos = model.rope_cos.to(device)
            sin = model.rope_sin.to(device)
            
            for blk in model.blocks:
                x = blk(x, cos, sin)
                primal_states.append(x)
            
            # Forward pass just to calculate standard Loss for your logging
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                x_norm_log = model.final_norm(x)
                logits_log = model.lm_head(x_norm_log)
                loss_val = F.cross_entropy(logits_log.view(-1, logits_log.size(-1)).float(), yb.view(-1), ignore_index=-100)
            step_loss += loss_val.item() / train_cfg.grad_accum_steps
            
            del x_norm_log, logits_log, loss_val

        # -------------------------------------------------------------------
        # 2. DUAL BOUNDARY CONDITION (Memory-Optimized Fused Kernel)
        # -------------------------------------------------------------------
        # Step back through Final Norm and LM Head locally
        x_last = primal_states[-1].detach().requires_grad_(True)
        
        with torch.enable_grad():
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                x_norm_val = model.final_norm(x_last)
                logits_val = model.lm_head(x_norm_val)

            boundary_loss = F.cross_entropy(
                logits_val.view(-1, logits_val.size(-1)).float(), 
                yb.view(-1), 
                ignore_index=-100
            )
            
            # Scale for gradient accumulation and Mixed Precision (FP16)
            scale_factor = 1.0 / train_cfg.grad_accum_steps
            if AMP_DTYPE == torch.float16:
                scale_factor *= scaler.get_scale()
                
            boundary_gap_proxy = boundary_loss * scale_factor
        
        # Calculate local dual momentum (d_current) and parameter gradients for the head
        head_params = list(model.final_norm.parameters()) + list(model.lm_head.parameters())
        
        # Extract the momentum (d_current) passing backwards functionally
        head_grads = torch.autograd.grad(boundary_gap_proxy, [x_last] + head_params)
        
        d_current = head_grads[0] # The momentum entering the top transformer block
        
        # Accumulate head gradients
        for p, g in zip(head_params, head_grads[1:]):
            if p.grad is None: p.grad = g.clone()
            else: p.grad += g

        # -------------------------------------------------------------------
        # 3. 
        # -------------------------------------------------------------------
        for i in reversed(range(len(model.blocks))):
            blk = model.blocks[i]
            
            # Retrieve the primal input state for this isolated block
            x_in = primal_states[i].detach().requires_grad_(True)
            
            with torch.enable_grad():
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    y_local = blk(x_in, cos, sin)
            
            # The Local Fenchel-Young Gap: Mapping Primal y to Dual d
            local_gap = torch.sum(y_local * d_current.detach())
            
            local_params = [p for p in blk.parameters() if p.requires_grad]
            
            # Compute exactly how weights must shift to close the duality gap,
            # and what momentum (d_in) gets passed to the next block.
            block_grads = torch.autograd.grad(local_gap, [x_in] + local_params)
            
            d_current = block_grads[0] # Pass momentum down
            
            for p, g in zip(local_params, block_grads[1:]):
                if p.grad is None: p.grad = g.clone()
                else: p.grad += g

        # -------------------------------------------------------------------
        # 4. FINAL EMBEDDING MATCHING
        # -------------------------------------------------------------------
        with torch.enable_grad():
            x_emb = model.tok_emb(xb)
            # The gap proxy for the embedding matrix
            emb_gap = torch.sum(x_emb * d_current.detach())
            
            emb_params = [model.tok_emb.weight]
            emb_grads = torch.autograd.grad(emb_gap, emb_params)
            
            for p, g in zip(emb_params, emb_grads):
                if p.grad is None: p.grad = g.clone()
                else: p.grad += g

    # -------------------------------------------------------------------
    # OPTIMIZER STEP (Identical to baseline, handling FP16 scale properly)
    # -------------------------------------------------------------------
    if AMP_DTYPE == torch.float16:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        optimizer.step()

    running_loss += step_loss

    # -------------------------------------------------------------------
    # LOGGING AND EVALUATION
    # -------------------------------------------------------------------
    if step % train_cfg.log_every == 0:
        elapsed = time.time() - t0
        avg_loss = running_loss / train_cfg.log_every if step > 0 else step_loss
        tokens_seen = (step + 1) * train_cfg.micro_batch_size * train_cfg.grad_accum_steps * model_cfg.max_seq_len
        toks_per_sec = tokens_seen / elapsed if elapsed > 0 else 0
        ppl = math.exp(min(avg_loss, 20))
        
        peak_vram_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
        
        print(f"step {step:5d} | loss {avg_loss:.4f} | ppl {ppl:8.2f} | lr {lr:.2e} "
              f"| tok/s {toks_per_sec:,.0f} | elapsed {elapsed/60:.1f}m | peak VRAM {peak_vram_gb:.2f} GB")
              
        history["step"].append(step)
        history["train_loss"].append(avg_loss)
        history["peak_vram_gb"].append(peak_vram_gb) # This fixes the (0,) error
        running_loss = 0.0
        
        torch.cuda.reset_peak_memory_stats(device)

    if step > 0 and step % train_cfg.eval_every == 0:
        eval_loss, eval_ppl = evaluate()
        print(f"          [eval @ step {step}] loss {eval_loss:.4f} | ppl {eval_ppl:8.2f}")
        history["eval_step"].append(step)
        history["eval_loss"].append(eval_loss)
        history["eval_ppl"].append(eval_ppl)

    if step > 0 and step % train_cfg.sample_every == 0:
        sample()

    if step > 0 and step % train_cfg.save_every == 0:
        save_checkpoint(step)

save_checkpoint(train_cfg.max_steps)
final_eval_loss, final_eval_ppl = evaluate()
print(f"Training complete. Final eval loss {final_eval_loss:.4f} | ppl {final_eval_ppl:.2f}")
sample()

# ============================================================================
# Final save
# ============================================================================

final_dir = "/content/final_model"
os.makedirs(final_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(final_dir, "model.pt"))
with open(os.path.join(final_dir, "model_config.json"), "w") as f:
    json.dump(asdict(model_cfg), f, indent=2)
tokenizer.save_pretrained(final_dir)

print(f"Saved to {final_dir}:")
for fn in os.listdir(final_dir):
    print(" -", fn)

# --- reload sanity check ---
reloaded_cfg = ModelConfig(**json.load(open(os.path.join(final_dir, "model_config.json"))))
reloaded_model = SOTALM(reloaded_cfg).to(device)
reloaded_model.load_state_dict(torch.load(os.path.join(final_dir, "model.pt"), map_location=device))
print("Reload OK.")

# # --- loss curve ---

# try:
#     import matplotlib.pyplot as plt
#     fig, ax = plt.subplots(figsize=(8, 4))
#     ax.plot(history["step"], history["train_loss"], label="train loss")
#     if history["eval_step"]:
#         ax.plot(history["eval_step"], history["eval_loss"], label="eval loss", marker="o")
#     ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("Training curve")
#     plt.savefig(os.path.join(final_dir, "loss_curve.png"), dpi=120, bbox_inches="tight")
#     plt.show()
# except Exception as e:
#     print(f"Skipping plot ({e})")

try:
    import matplotlib.pyplot as plt
    fig, ax1 = plt.subplots(figsize=(8, 4))
    
    # Plot losses on primary axis
    if len(history["step"]) > 0:
        ax1.plot(history["step"], history["train_loss"], label="train loss", color="tab:blue", marker="o")
    if len(history["eval_step"]) > 0:
        ax1.plot(history["eval_step"], history["eval_loss"], label="eval loss", marker="s", color="tab:cyan")
        
    ax1.set_xlabel("step")
    ax1.set_ylabel("loss", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    
    # Plot VRAM on secondary axis
    if len(history["step"]) > 0:
        ax2 = ax1.twinx()
        ax2.plot(history["step"], history["peak_vram_gb"], label="peak VRAM (GB)", color="tab:red", linestyle="--", marker="x")
        ax2.set_ylabel("peak VRAM (GB)", color="tab:red")
        ax2.tick_params(axis="y", labelcolor="tab:red")
        
    # --- ADDED: Combine legends into a single box ---
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    
    # Check if ax2 was created before trying to get its legend
    if 'ax2' in locals():
        lines_2, labels_2 = ax2.get_legend_handles_labels()
    else:
        lines_2, labels_2 = [], []
        
    # Combine the lines and labels from both axes
    all_lines = lines_1 + lines_2
    all_labels = labels_1 + labels_2
    
    # Draw one unified legend on the upper left
    if all_lines:
        ax1.legend(all_lines, all_labels, loc="upper left")
    
    plt.title("Training Curve and VRAM Usage")
    plt.savefig(os.path.join(final_dir, "loss_and_vram_curve.png"), dpi=120, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Skipping plot ({e})")